# 01 — Parsing & Split — AI4I 2020

Pipeline de préparation **avant entraînement** (tout est visible cellule par cellule) :

1. Chargement brut + contrôles qualité visibles (dtypes, NaN, doublons) ;
2. Exclusion des **colonnes de fuite** (`TWF/HDF/PWF/OSF/RNF/failure_type` composent ou dérivent la cible) ;
3. Nettoyage typé (`Type` → category, cible binaire vérifiée) ;
4. **Split stratifié train/test 80/20** (seed 42) → export sur disque.

> La normalisation (imputation + StandardScaler) et le feature engineering
> (capping, dérivées) sont volontairement **PAS appliqués ici** : ils restent
> dans le pipeline d'entraînement (notebook 02), sérialisés avec le modèle,
> pour que l'API reçoive des capteurs bruts sans double transformation.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

TARGET = "machine_failure"
LEAKAGE_COLS = ["TWF", "HDF", "PWF", "OSF", "RNF", "failure_type"]
TEST_SIZE = 0.2
SEED = 42


def locate_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if candidate.name == "GMAO-ML":
            return candidate
        if (candidate / "GMAO-ML" / "data").is_dir():
            return candidate / "GMAO-ML"
        if (candidate / "data" / "ai4i_2020.csv").is_file():
            return candidate
    raise FileNotFoundError("Dossier GMAO-ML/ introuvable")


PROJECT_DIR = locate_project_dir()
DATA_DIR = PROJECT_DIR / "data"
RAW_PATH = DATA_DIR / "ai4i_2020.csv"
print(f"Projet : {PROJECT_DIR}")

Projet : /home/abdellah-daif/Gmao-Services-IA/GMAO-ML


## 1. Chargement brut

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f"Shape : {df_raw.shape}")
df_raw.head()

Shape : (10000, 13)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],machine_failure,TWF,HDF,PWF,OSF,RNF,failure_type
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,No Failure
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,No Failure
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,No Failure
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,No Failure
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,No Failure


## 2. Contrôles qualité visibles

In [3]:
quality = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_nan": df_raw.isna().sum(),
    "pct_nan": (df_raw.isna().mean() * 100).round(2),
    "n_uniques": df_raw.nunique(),
})
display(quality)
print(f"Lignes dupliquées : {df_raw.duplicated().sum()}")

,dtype,n_nan,pct_nan,n_uniques
Type,object,0,0.0,3
Air temperature [K],float64,0,0.0,93
Process temperature [K],float64,0,0.0,82
Rotational speed [rpm],int64,0,0.0,941
Torque [Nm],float64,0,0.0,577
Tool wear [min],int64,0,0.0,246
machine_failure,int64,0,0.0,2
TWF,int64,0,0.0,2
HDF,int64,0,0.0,2
PWF,int64,0,0.0,2


Lignes dupliquées : 0


## 3. Cible : cohérence & équilibre

In [4]:
assert df_raw[TARGET].isin([0, 1]).all(), "La cible doit être binaire 0/1"
balance = df_raw[TARGET].value_counts().rename("count").to_frame()
balance["pct"] = (balance["count"] / len(df_raw) * 100).round(2)
display(balance)
print(f"Taux de panne global : {df_raw[TARGET].mean() * 100:.2f} %")

,count,pct
machine_failure,,
0,9661,96.61
1,339,3.39


Taux de panne global : 3.39 %


## 4. Exclusion des colonnes de fuite

`TWF/HDF/PWF/OSF/RNF` sont les modes de défaillance qui **composent**
`machine_failure`, et `failure_type` en est la forme catégorielle.
Les garder = métriques parfaites et fausses (~100 %).

In [5]:
present_leakage = [c for c in LEAKAGE_COLS if c in df_raw.columns]
print(f"Fuites détectées dans le fichier : {present_leakage}")

df_clean = df_raw.drop(columns=present_leakage)
print(f"Shape après exclusion : {df_clean.shape}")
print(f"Colonnes conservées   : {list(df_clean.columns)}")

Fuites détectées dans le fichier : ['TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'failure_type']
Shape après exclusion : (10000, 7)
Colonnes conservées   : ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'machine_failure']


## 5. Nettoyage typé

In [6]:
df_clean["Type"] = df_clean["Type"].astype("category")

numeric_cols = [c for c in df_clean.select_dtypes(include="number").columns if c != TARGET]
df_clean[numeric_cols] = df_clean[numeric_cols].apply(pd.to_numeric)

print("Dtypes finaux :")
print(df_clean.dtypes.to_string())
print(f"\nNaN restants : {int(df_clean.isna().sum().sum())}")

Dtypes finaux :
Type                       category
Air temperature [K]         float64
Process temperature [K]     float64
Rotational speed [rpm]        int64
Torque [Nm]                 float64
Tool wear [min]               int64
machine_failure               int64

NaN restants : 0


## 6. Split stratifié train / test (80/20)

Le stratified split garantit le même taux de panne (~3.4 %) dans les deux
jeux — crucial vu le déséquilibre.

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df_clean[TARGET],
)

split_recap = pd.DataFrame({
    "lignes": [len(train_df), len(test_df)],
    "pannes": [int(train_df[TARGET].sum()), int(test_df[TARGET].sum())],
    "taux_panne_%": [
        round(train_df[TARGET].mean() * 100, 2),
        round(test_df[TARGET].mean() * 100, 2),
    ],
}, index=["train", "test"])
display(split_recap)

assert set(train_df.index) & set(test_df.index) == set(), "Chevauchement train/test !"
print("Aucun chevauchement d'index entre train et test.")

,lignes,pannes,taux_panne_%
train,8000,271,3.39
test,2000,68,3.40


Aucun chevauchement d'index entre train et test.


## 7. Export vers `data/`

Ces deux fichiers sont l'entrée du notebook 02 (entraînement + évaluation).

In [8]:
TRAIN_OUT = DATA_DIR / "train_ai4i_2020.csv"
TEST_OUT = DATA_DIR / "test_ai4i_2020.csv"

train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

print("=" * 60)
print("EXPORTS TERMINÉS")
print("=" * 60)
print(f"Train : {TRAIN_OUT.name:<22} {train_df.shape[0]:>6} lignes × {train_df.shape[1]} colonnes")
print(f"Test  : {TEST_OUT.name:<22} {test_df.shape[0]:>6} lignes × {test_df.shape[1]} colonnes")
print("-" * 60)
print("Étape suivante : notebooks/02_training_evaluation_ai4i.ipynb")

EXPORTS TERMINÉS
Train : train_ai4i_2020.csv      8000 lignes × 7 colonnes
Test  : test_ai4i_2020.csv       2000 lignes × 7 colonnes
------------------------------------------------------------
Étape suivante : notebooks/02_training_evaluation_ai4i.ipynb
